<a href="https://colab.research.google.com/github/liangchow/sade-geo/blob/main/Module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Set Up Worksheet and Import Libraries

In [ ]:
# Clone Gitub repository to Colab
from google.colab import drive
drive.mount('/content/drive')

!apt-get install git
!git clone https://github.com/liangchow/sade-geo.git

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
fatal: destination path 'sade-geo' already exists and is not an empty directory.


In [ ]:
import requests
import json
from pyproj import Transformer

In [ ]:
# Hazard URLs
LIQUEFACTION_URL = (
    "https://services2.arcgis.com/zr3KAIbsRSUyARHG/"
    "arcgis/rest/services/CGS_Liquefaction_Zones/"
    "FeatureServer/0/query"
)

AP_FAULT_URL = (
    "https://services2.arcgis.com/zr3KAIbsRSUyARHG/"
    "arcgis/rest/services/"
    "CGS_Alquist_Priolo_Fault_Zones/"
    "FeatureServer/0/query"
)

LANDSLIDE_URL = (
    "https://services2.arcgis.com/zr3KAIbsRSUyARHG/"
    "arcgis/rest/services/"
    "CGS_Landslide_Zones/"
    "FeatureServer/0/query"
)

UNEVALUATED_URL = (
    "https://services2.arcgis.com/zr3KAIbsRSUyARHG/"
    "arcgis/rest/services/"
    "CGS_SHZ_Unevaluated_Areas/"
    "FeatureServer/0/query"
)

GEOLOGY_URL = (
    "https://gis.conservation.ca.gov/server/rest/services/"
    "CGS/Geologic_Map_of_California/"
    "MapServer/12/query"
)

VS30_URL = (
    "https://gis.conservation.ca.gov/server/rest/services/"
    "CGS/MS48_Vs30_ShearWaveVelocity2022/"
    "ImageServer"
)

GIS_LAYERS = {
    "Liq": {"url": LIQUEFACTION_URL, "type": "boolean"},
    "AP Fault": {"url": AP_FAULT_URL, "type": "boolean"},
    "Landslide": {"url": LANDSLIDE_URL, "type": "boolean"},
    "Unevaluated": {"url": UNEVALUATED_URL, "type": "boolean"},
    "Geology": {"url": GEOLOGY_URL, "type": "attributes", "query_function": "geology",
        "fields": [
            "PTYPE",
            "GENERAL_LITHOLOGY",
            "AGE",
            "DESCRIPTION"
        ]},
    "Vs30": {"url": VS30_URL, "type": "raster"},
}

In [ ]:
# FeatureServer query function
def query_feature_service(url, lat, lon, inSR=4326):

    params = {
        "f": "json",
        "geometry": f"{lon},{lat}",
        "geometryType": "esriGeometryPoint",
        "spatialRel": "esriSpatialRelIntersects",
        "inSR": inSR,
        "returnGeometry": "false",
        "outFields": "*"
    }

    r = requests.get(url, params=params)
    r.raise_for_status()

    return r.json()


def get_layer_result(layer, lat, lon):
    # Raster layers (ImageServer)
    if layer["type"] == "raster":
        return query_vs30(lat, lon)

    # Special geometry handling for geology
    if layer.get("query_function") == "geology":
        result = query_geology(lat, lon)
    # Normal FeatureServer layers
    else:
        result = query_feature_service(
            layer["url"], lat, lon
        )
    features = result.get("features", [])

    if layer["type"] == "boolean":
        return len(features) > 0

    elif layer["type"] == "attributes":
        if not features:
            return None
        attrs = features[0]["attributes"]
        return {
            field: attrs.get(field)
            for field in layer["fields"]
        }

    else:
        return None

def get_all_services(lat, lon):

  results = {}

  for name, layer in GIS_LAYERS.items():
      results[name] = get_layer_result(
          layer,
          lat,
          lon
      )

  return results

In [ ]:
# Geology MapServer query function
to_webmercator = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3857",
    always_xy=True
)

def query_geology(lat, lon):

    x, y = to_webmercator.transform(lon, lat)

    params = {
        "f": "json",
        "geometry": f"{x},{y}",
        "geometryType": "esriGeometryPoint",
        "spatialRel": "esriSpatialRelIntersects",
        "inSR": 102100,
        "returnGeometry": "false",
        "outFields": (
            "OBJECTID,"
            "PTYPE,"
            "GENERAL_LITHOLOGY,"
            "AGE,"
            "DESCRIPTION"
        )
    }

    r = requests.get(GEOLOGY_URL, params=params)
    r.raise_for_status()

    return r.json()

In [ ]:
# Vs30 ImageServer query function
def query_vs30(lat, lon):

    params = {
        "f": "json",
        "geometry": f"{lon},{lat}",
        "geometryType": "esriGeometryPoint",
        "returnGeometry": "false"
    }

    r = requests.get(
        VS30_URL + "/identify",
        params=params
    )

    r.raise_for_status()

    data = r.json()

    return data.get("value")

##Test Cell

In [ ]:
# Example lat/long
lat = 37.7749
lon = -122.4194

get_all_services(lat, lon)

{'Liq': True,
 'AP Fault': False,
 'Landslide': False,
 'Unevaluated': False,
 'Geology': {'PTYPE': 'Qs',
  'GENERAL_LITHOLOGY': 'marine and nonmarine (continental) sedimentary rocks',
  'AGE': 'Pleistocene-Holocene',
  'DESCRIPTION': 'Extensive marine and nonmarine sand deposits, generally near the coast or desert playas.'},
 'Vs30': '225.6'}

### Geological Hazard Assessment

Based on the provided latitude (`37.7749`) and longitude (`-122.4194`):

- **Liquefaction (Liq):** `True` (Presence of liquefaction zones)
- **Alquist-Priolo Fault (AP Fault):** `False` (No Alquist-Priolo Fault zones detected)
- **Landslide:** `False` (No landslide zones detected)
- **Unevaluated Areas:** `False` (Area has been evaluated)
- **Geology:**
  - **PTYPE:** `Qs`
  - **GENERAL_LITHOLOGY:** `marine and nonmarine (continental) sedimentary rocks`
  - **AGE:** `Pleistocene-Holocene`
  - **DESCRIPTION:** `Extensive marine and nonmarine sand deposits, generally near the coast or desert playas.`
- **Vs30 (Shear-wave velocity):** `225.6` (meters/second)